# Financial Sentiment Project: Experiment Runner

## 1. Environment And Project Root
This cell detects the repository root, switches the working directory there, and prepares helper functions used throughout the notebook.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import platform
from typing import Iterable

import pandas as pd
import yaml

def find_project_root(start: Path | None = None) -> Path:
    sentinels = ("README.md", "requirements.txt")
    current = (start or Path.cwd()).resolve()
    candidates = [current] + list(current.parents)
    for candidate in candidates:
        if all((candidate / s).exists() for s in sentinels):
            return candidate
    raise FileNotFoundError("Could not locate project root from current notebook location.")

ROOT = find_project_root()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

IN_COLAB = "google.colab" in sys.modules

print(f"Project root: {ROOT}")
print(f"Python: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"In Colab: {IN_COLAB}")
print(f"Current working directory: {Path.cwd()}")

## 2. Optional: Mount Google Drive In Colab

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
else:
    print("Not running in Colab. Skipping Drive mount.")

## 3. Optional: Install Dependencies

In [ ]:
cmd = [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")]
print("$", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT), text=True)
if result.returncode != 0:
    raise RuntimeError("Dependency installation failed.")
print("Dependencies are ready.")

## 4. Helper Functions
Funtions :
- run project commands
- detect newly created outputs after timestamped runs
- read and flatten metrics JSON files

In [ ]:
def run_command(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env["PROJECT_ROOT"] = str(ROOT)

    cwd = cwd or ROOT
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        env=env,
        text=True,
        capture_output=True,
    )

    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def snapshot_paths(pattern: str) -> set[str]:
    return {str(p.resolve()) for p in ROOT.glob(pattern)}


def newly_created_paths(before: set[str], pattern: str) -> list[Path]:
    after = snapshot_paths(pattern)
    new_items = [Path(p) for p in (after - before)]
    return sorted(new_items, key=lambda p: p.stat().st_mtime)


def newest_path(pattern: str) -> Path | None:
    matches = sorted(ROOT.glob(pattern), key=lambda p: p.stat().st_mtime)
    return matches[-1] if matches else None


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def flatten_metrics_json(metrics_path: Path) -> pd.DataFrame:
    payload = load_json(metrics_path)

    model_name = payload.get("model_name") or payload.get("model_type") or "unknown"
    run_name = payload.get("run_name", "")
    train_path = payload.get("train_path")
    train_domain = Path(train_path).parent.name if train_path else "none"

    rows = []
    for eval_name, eval_payload in payload.get("eval_sets", {}).items():
        eval_path = eval_payload.get("path", "")
        eval_domain = Path(eval_path).parent.name if eval_path else "unknown"
        eval_split = Path(eval_path).stem if eval_path else eval_name

        rows.append(
            {
                "run_name": run_name,
                "model": model_name,
                "train_domain": train_domain,
                "eval_name": eval_name,
                "eval_domain": eval_domain,
                "eval_split": eval_split,
                "setting": (
                    "lexicon_eval"
                    if train_domain == "none"
                    else "in_domain" if train_domain == eval_domain else "cross_domain"
                ),
                "accuracy": eval_payload.get("accuracy"),
                "macro_f1": eval_payload.get("macro_f1"),
                "weighted_f1": eval_payload.get("weighted_f1"),
                "metrics_path": str(metrics_path),
                "prediction_path": eval_payload.get("prediction_path"),
                "confusion_matrix_path": eval_payload.get("confusion_matrix_path"),
                "confusion_figure_path": eval_payload.get("confusion_figure_path"),
            }
        )

    return pd.DataFrame(rows)


def collect_all_metrics() -> pd.DataFrame:
    metric_files = []
    for path in ROOT.glob("outputs/metrics/**/*.json"):
        try:
            payload = load_json(path)
            if "eval_sets" in payload:
                metric_files.append(path)
        except Exception:
            continue

    frames = [flatten_metrics_json(path) for path in metric_files]
    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["model", "run_name", "eval_name"]).reset_index(drop=True)
    return combined


def display_if_exists(path: Path):
    if path.exists():
        if path.suffix.lower() == ".csv":
            display(pd.read_csv(path))
        elif path.suffix.lower() == ".json":
            display(pd.DataFrame(load_json(path)))
        else:
            print(path)
    else:
        print(f"Not found: {path}")

## 5. Check Required Files

In [ ]:
required_files = [
    ROOT / "configs/baseline.yaml",
    ROOT / "configs/baseline_tfns.yaml",
    ROOT / "configs/transformer_bert_fpb.yaml",
    ROOT / "configs/transformer_bert_tfns.yaml",
    ROOT / "configs/transformer_finbert_fpb.yaml",
    ROOT / "configs/transformer_finbert_tfns.yaml",
    ROOT / "configs/mlm_pretrain.yaml",
    ROOT / "configs/transformer_finbert_mlm_fpb.yaml",
    ROOT / "configs/transformer_finbert_mlm_tfns.yaml",
    ROOT / "configs/rule_based.yaml",
    ROOT / "data/processed/fpb/train.csv",
    ROOT / "data/processed/fpb/val.csv",
    ROOT / "data/processed/fpb/test.csv",
    ROOT / "data/processed/tfns/train.csv",
    ROOT / "data/processed/tfns/val.csv",
    ROOT / "data/processed/tfns/test.csv",
    ROOT / "data/lexicon/Loughran-McDonald_MasterDictionary_1993-2025.csv",
]

status_rows = []
for path in required_files:
    status_rows.append({"path": str(path.relative_to(ROOT)), "exists": path.exists()})

status_df = pd.DataFrame(status_rows)
display(status_df)

missing = status_df.loc[~status_df["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")
else:
    print("All required files are present.")

## 6. Rule-Based Baseline
This runs the Loughran-McDonald lexicon baseline using `configs/rule_based.yaml`.
A new timestamped metrics JSON file will be created under `outputs/metrics/rule_based_baseline/`.

In [ ]:
before = snapshot_paths("outputs/metrics/rule_based_baseline/*.json")

run_command([
    sys.executable,
    "-m",
    "src.cli.run_rule_based_baseline",
    "--config",
    str(ROOT / "configs/rule_based.yaml"),
])

new_metric_files = newly_created_paths(before, "outputs/metrics/rule_based_baseline/*.json")
rule_metrics_path = new_metric_files[-1] if new_metric_files else newest_path("outputs/metrics/rule_based_baseline/*.json")

print(f"Rule-based metrics file: {rule_metrics_path}")
rule_df = flatten_metrics_json(rule_metrics_path)
display(rule_df)

## 7. Statistical Baselines
This runs the batch script for TF-IDF + Logistic Regression, Linear SVM, and Naive Bayes.
The batch summary is expected at `outputs/metrics/statistical_baselines/summary.csv`.

In [ ]:
run_command([
    sys.executable,
    str(ROOT / "scripts/run_statistical_baselines.py"),
])

stat_summary_path = ROOT / "outputs/metrics/statistical_baselines/summary.csv"
print(f"Statistical baseline summary: {stat_summary_path}")
display(pd.read_csv(stat_summary_path))

## 8. Transformer FT-Only Baselines
This runs the configured transformer baselines.
GPU is strongly recommended.
The batch summary is expected at `outputs/metrics/transformer_baselines/summary.csv`.

In [ ]:
transformer_configs = [
    str(ROOT / "configs/transformer_bert_fpb.yaml"),
    str(ROOT / "configs/transformer_bert_tfns.yaml"),
    str(ROOT / "configs/transformer_finbert_fpb.yaml"),
    str(ROOT / "configs/transformer_finbert_tfns.yaml"),
]

run_command([
    sys.executable,
    str(ROOT / "scripts/run_transformer_baselines.py"),
    "--configs",
    *transformer_configs,
])

transformer_summary_path = ROOT / "outputs/metrics/transformer_baselines/summary.csv"
print(f"Transformer summary: {transformer_summary_path}")
display(pd.read_csv(transformer_summary_path))

## 9. Continued MLM Pretraining
This runs the MLM pretraining stage defined in `configs/mlm_pretrain.yaml`.
A new checkpoint directory ending with `_best` should be created under `outputs/checkpoints/mlm_pretrain/`.

In [ ]:
before_ckpts = snapshot_paths("outputs/checkpoints/mlm_pretrain/*_best")

run_command([
    sys.executable,
    "-m",
    "src.cli.run_mlm_pretrain",
    "--config",
    str(ROOT / "configs/mlm_pretrain.yaml"),
])

new_ckpts = newly_created_paths(before_ckpts, "outputs/checkpoints/mlm_pretrain/*_best")
mlm_checkpoint_dir = new_ckpts[-1] if new_ckpts else newest_path("outputs/checkpoints/mlm_pretrain/*_best")

print(f"MLM checkpoint directory: {mlm_checkpoint_dir}")
if mlm_checkpoint_dir is None:
    raise FileNotFoundError("Could not find MLM checkpoint directory after pretraining.")

## 10. Create Temporary MLM Fine-Tuning Configs
The original MLM fine-tuning configs contain placeholder checkpoint paths.
This cell creates temporary configs under `outputs/tmp_configs/` without modifying the original files.

In [ ]:
def make_temp_mlm_ft_config(src_config_path: Path, checkpoint_dir: Path, suffix: str) -> Path:
    with src_config_path.open("r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    config["model"]["name"] = str(checkpoint_dir)

    out_dir = ROOT / "outputs/tmp_configs"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{src_config_path.stem}_{suffix}.yaml"
    with out_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

    print(f"Temporary config written to: {out_path}")
    return out_path

if "mlm_checkpoint_dir" not in globals() or mlm_checkpoint_dir is None:
    mlm_checkpoint_dir = newest_path("outputs/checkpoints/mlm_pretrain/*_best")

if mlm_checkpoint_dir is None:
    raise FileNotFoundError("No MLM checkpoint found. Run the MLM pretraining cell first, or set mlm_checkpoint_dir manually.")

temp_mlm_fpb_config = make_temp_mlm_ft_config(
    ROOT / "configs/transformer_finbert_mlm_fpb.yaml",
    mlm_checkpoint_dir,
    "auto",
)

temp_mlm_tfns_config = make_temp_mlm_ft_config(
    ROOT / "configs/transformer_finbert_mlm_tfns.yaml",
    mlm_checkpoint_dir,
    "auto",
)

print("Temporary MLM fine-tuning configs are ready.")
print(temp_mlm_fpb_config)
print(temp_mlm_tfns_config)

## 11. MLM + Fine-Tuning On FPB
This runs transformer fine-tuning using the MLM checkpoint on the FPB training split.

In [ ]:
before = snapshot_paths("outputs/metrics/mlm_finetune/*.json")

run_command([
    sys.executable,
    "-m",
    "src.cli.train_transformer",
    "--config",
    str(temp_mlm_fpb_config),
])

new_metric_files = newly_created_paths(before, "outputs/metrics/mlm_finetune/*.json")
mlm_fpb_metrics_path = new_metric_files[-1] if new_metric_files else newest_path("outputs/metrics/mlm_finetune/*.json")

print(f"MLM+FT (FPB) metrics file: {mlm_fpb_metrics_path}")
display(flatten_metrics_json(mlm_fpb_metrics_path))

## 12. MLM + Fine-Tuning On TFNS
This runs transformer fine-tuning using the MLM checkpoint on the TFNS training split.

In [ ]:
before = snapshot_paths("outputs/metrics/mlm_finetune/*.json")

run_command([
    sys.executable,
    "-m",
    "src.cli.train_transformer",
    "--config",
    str(temp_mlm_tfns_config),
])

new_metric_files = newly_created_paths(before, "outputs/metrics/mlm_finetune/*.json")
mlm_tfns_metrics_path = new_metric_files[-1] if new_metric_files else newest_path("outputs/metrics/mlm_finetune/*.json")

print(f"MLM+FT (TFNS) metrics file: {mlm_tfns_metrics_path}")
display(flatten_metrics_json(mlm_tfns_metrics_path))

## 13. Collect And Summarize All Metrics

In [ ]:
all_results_df = collect_all_metrics()

if all_results_df.empty:
    print("No metrics JSON files found under outputs/metrics.")
else:
    display(all_results_df)

summary_out = ROOT / "outputs/metrics/experiment_manifest.csv"
summary_out.parent.mkdir(parents=True, exist_ok=True)
all_results_df.to_csv(summary_out, index=False)
print(f"Combined experiment manifest saved to: {summary_out}")